# Phase 2 train — Simple-BEV recipe + наши фиксы

Что нового по сравнению с train_v1:
1. **Group-aware val split** по `(rover, ride_date)` — val ≈ test distribution (фикс блокера)
2. **Encoder UNFROZEN** + 2 param groups (lr backbone 1e-5, lr head 3e-4)
3. **IMG_HW = 448×800** — paper baseline
4. **Gradient accumulation** — effective batch ~32 (paper показал +14 IoU vs batch 2)
5. **Random resize+crop aug** с пересчётом intrinsic per-camera (+1.6 IoU per paper)
6. **Lovasz loss** (0.5 BCE + 0.3 Dice + 0.2 Lovasz) — прямо оптимизирует IoU
7. **EMA weights** — финальная модель = EMA копия
8. Без camera dropout (paper: −1 IoU)
9. Без photometric aug (paper: 0 IoU)

## Прогноз
От baseline 0.51 → **0.57-0.62 на test** (учитывая что val теперь группированный, должен быть честнее)

## Бюджет
- T4: ~6-10 часов (1 запуск)
- A100: ~2-4 часа (1 запуск, рекомендую этот вариант)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from pathlib import Path
import time, csv, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import matplotlib.pyplot as plt

from bev_v1 import compute_pos_weight, compute_coverage_weights, iou_binary_batch, BEV_H, BEV_W
from bev_v2 import MultiCamBEVPretrainedEncoder
from bev_v3 import BEVDatasetAug, CompoundLossV2, make_group_aware_split

device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print(f"device = {device}")

DATA_TRAIN = Path("./autonomy_yandex_dataset_train/")
DATA_VAL = Path("./autonomy_yandex_dataset_val/")
DATA_TEST = Path("./autonomy_yandex_dataset_test/")

## 1. Конфиг (выбираешь PRESET под железо)

In [2]:
# Меняй под целевое железо
PRESET = "A100_PHASE2"

CONFIGS = {
    "SMOKE": dict(
        epochs=2, img_hw=(256, 512), batch_size=2, grad_accum=1,
        num_workers=0, lr_backbone=1e-5, lr_head=3e-4,
        use_aug=False, use_amp=False, ema_decay=0.0,
        use_group_split=True, val_holdout_frac=0.2,
        use_official_val=False,  # для smoke используем group split
        use_sampler=False,
    ),
    "T4_PHASE2": dict(
        epochs=20, img_hw=(384, 704), batch_size=4, grad_accum=8,
        num_workers=4, lr_backbone=1e-5, lr_head=3e-4,
        use_aug=True, use_amp=True, ema_decay=0.999,
        use_group_split=True, val_holdout_frac=0.2,
        use_official_val=False,
        use_sampler=True,
    ),
    "A100_PHASE2": dict(
        epochs=25, img_hw=(448, 800), batch_size=8, grad_accum=4,
        num_workers=6, lr_backbone=1e-5, lr_head=3e-4,
        use_aug=True, use_amp=True, ema_decay=0.999,
        use_group_split=True, val_holdout_frac=0.2,
        use_official_val=False,
        use_sampler=True,
    ),
}
cfg = CONFIGS[PRESET]
print(f"PRESET = {PRESET}")
for k, v in cfg.items():
    print(f"  {k} = {v}")
print(f"  effective batch = {cfg['batch_size'] * cfg['grad_accum']}")

RUN_NAME = f"bev_v3_{PRESET.lower()}"
OUT_DIR = Path("./runs") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT_DIR / "log.csv"
CKPT_BEST = OUT_DIR / "best.pt"
CKPT_LAST = OUT_DIR / "last.pt"
CKPT_EMA = OUT_DIR / "ema.pt"
print(f"Run dir: {OUT_DIR.resolve()}")

PRESET = A100_PHASE2
  epochs = 25
  img_hw = (448, 800)
  batch_size = 8
  grad_accum = 4
  num_workers = 6
  lr_backbone = 1e-05
  lr_head = 0.0003
  use_aug = True
  use_amp = True
  ema_decay = 0.999
  use_group_split = True
  val_holdout_frac = 0.2
  use_official_val = False
  use_sampler = True
  effective batch = 32
Run dir: /home/jupyter/project/runs/bev_v3_a100_phase2


## 2. Group-aware split

Раньше: train, val (отдельный split) — но между ними оказались общие группы → val переобучен.
Теперь: берём официальный train (4000 семплов) и **внутри него** делим на train/val по группам `(rover, ride_date)` так, чтобы группы не пересекались. Имитируем то что test = новые группы.

Официальный val (1000 семплов) используем как **дополнительный мониторинг** (отдельный вызов).

Если хочешь использовать только официальный val — поставь `use_official_val=True` в cfg.

In [3]:
if cfg["use_group_split"] and not cfg["use_official_val"]:
    train_idx, val_idx = make_group_aware_split(
        DATA_TRAIN / "info.csv",
        group_cols=("rover", "ride_date"),
        holdout_frac=cfg["val_holdout_frac"],
        seed=42,
        cache_path=OUT_DIR / "group_split.npz",
    )
    # Train/val оба из официального train с group-split
    train_ds_full = BEVDatasetAug(DATA_TRAIN, mode="train", img_hw=cfg["img_hw"], aug=cfg["use_aug"])
    val_ds_full = BEVDatasetAug(DATA_TRAIN, mode="val", img_hw=cfg["img_hw"], aug=False)
    train_ds = Subset(train_ds_full, train_idx)
    val_ds = Subset(val_ds_full, val_idx)
    OFFICIAL_VAL_IDX = None  # не используем
else:
    # Старая схема — train с aug, official val без aug
    train_ds = BEVDatasetAug(DATA_TRAIN, mode="train", img_hw=cfg["img_hw"], aug=cfg["use_aug"])
    val_ds = BEVDatasetAug(DATA_VAL, mode="val", img_hw=cfg["img_hw"], aug=False)

# Дополнительно — official val для extra мониторинга (не используется для best.pt)
official_val_ds = BEVDatasetAug(DATA_VAL, mode="val", img_hw=cfg["img_hw"], aug=False)

print(f"train: {len(train_ds)}  val(group-holdout): {len(val_ds)}  official_val: {len(official_val_ds)}")

train: 3042  val(group-holdout): 958  official_val: 1000


## 3. Coverage-aware sampler + pos_weight

In [4]:
# Coverage weights считаем 1 раз на ВСЕМ train, потом фильтруем по train_idx
sampler = None
if cfg["use_sampler"]:
    weights_all = compute_coverage_weights(
        DATA_TRAIN / "info.csv",
        cache_path=Path("./runs/coverage_weights.npy"),
        alpha=0.5, min_weight=0.1,
    )
    if cfg["use_group_split"] and not cfg["use_official_val"]:
        weights = weights_all[np.array(train_idx)]
    else:
        weights = weights_all
    print(f"  sampler weights (post-filter): min={weights.min():.3f}, max={weights.max():.3f}, mean={weights.mean():.3f}")
    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(weights).double(),
        num_samples=len(weights), replacement=True,
    )

train_loader = DataLoader(
    train_ds, batch_size=cfg["batch_size"],
    shuffle=(sampler is None), sampler=sampler,
    num_workers=cfg["num_workers"], drop_last=True, pin_memory=True,
    persistent_workers=cfg["num_workers"] > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=cfg["batch_size"], shuffle=False,
    num_workers=cfg["num_workers"], pin_memory=True,
    persistent_workers=cfg["num_workers"] > 0,
)
official_val_loader = DataLoader(
    official_val_ds, batch_size=cfg["batch_size"], shuffle=False,
    num_workers=cfg["num_workers"], pin_memory=True,
)
print(f"train batches: {len(train_loader)}  val batches: {len(val_loader)}  official_val: {len(official_val_loader)}")

# pos_weight
if PRESET == "SMOKE":
    pos_weight = 8.0
else:
    print("Computing pos_weight from train (50 batches)...")
    pw_loader = DataLoader(BEVDatasetAug(DATA_TRAIN, "train", cfg["img_hw"], aug=False),
                           batch_size=8, shuffle=False, num_workers=2)
    pos_weight = compute_pos_weight(pw_loader, max_batches=50)
    print(f"  pos_weight = {pos_weight:.2f}")

  sampler weights (post-filter): min=0.100, max=0.761, mean=0.501
train batches: 380  val batches: 120  official_val: 125
Computing pos_weight from train (50 batches)...
  pos_weight = 1.20


## 4. Model — Simple-BEV Encoder_res101 + наш decoder

**UNFROZEN backbone**: lr backbone в 30 раз меньше чем у головы (warmup-style).
Если хочешь возобновить от чекпоинта — раскомментируй RESUME_FROM.

In [5]:
model = MultiCamBEVPretrainedEncoder(
    load_pretrained=True,
    freeze_encoder=False,  # ← UNFROZEN
).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params: total={n_total/1e6:.2f}M  trainable={n_trainable/1e6:.2f}M")

# 2 param groups: backbone (low lr) и rest (high lr)
backbone_params = list(model.backbone.parameters())
head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
optimizer = AdamW(
    [
        {"params": backbone_params, "lr": cfg["lr_backbone"], "weight_decay": 1e-4},
        {"params": head_params, "lr": cfg["lr_head"], "weight_decay": 1e-4},
    ],
)
n_grad_steps = (len(train_loader) // cfg["grad_accum"]) * cfg["epochs"]
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, n_grad_steps))
criterion = CompoundLossV2(
    pos_weight=pos_weight, weight_bce=0.5, weight_dice=0.3, weight_lovasz=0.2,
).to(device)

amp_enabled = cfg["use_amp"] and (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
print(f"AMP enabled: {amp_enabled}")
print(f"Total grad-update steps: {n_grad_steps}")

# === Resume опционально ===
RESUME_FROM = None  # пример: Path("./runs/bev_v3_t4_phase2/best.pt")
if RESUME_FROM and Path(RESUME_FROM).exists():
    ckpt = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(ckpt["model"], strict=False)
    print(f"resumed from {RESUME_FROM}, prev val_iou = {ckpt.get('val_iou')}")

# EMA — простое exponential moving average
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        for p in self.shadow.parameters():
            p.requires_grad_(False)
        self.shadow.eval()
    @torch.no_grad()
    def update(self, model):
        for s_p, p in zip(self.shadow.parameters(), model.parameters()):
            s_p.mul_(self.decay).add_(p.data, alpha=1 - self.decay)
        for s_b, b in zip(self.shadow.buffers(), model.buffers()):
            s_b.copy_(b)

ema = EMA(model, decay=cfg["ema_decay"]) if cfg["ema_decay"] > 0 else None
print(f"EMA: {'enabled' if ema else 'disabled'}")

if not LOG_PATH.exists():
    with open(LOG_PATH, "w") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "val_iou", "ema_val_iou", "official_val_iou", "lr_head", "time_s"])

history = {"train_loss": [], "val_iou": [], "ema_val_iou": [], "official_val_iou": []}
best_iou = 0.0

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /tmp/xdg_cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 62.3MB/s]
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /tmp/xdg_cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100%|██████████| 171M/171M [00:02<00:00, 69.8MB/s]

[bev_v2] loaded encoder weights from external/simple_bev/checkpoints/8x5_5e-4_rgb12_22:43:46/model-000025000.pth
  matched: 568 / 568 keys
params: total=39.01M  trainable=39.01M
AMP enabled: True
Total grad-update steps: 2375
EMA: enabled


## 5. Training loop с gradient accumulation

In [ ]:
def train_one_epoch(model, loader, opt, sched, criterion, device, scaler, amp_enabled,
                    grad_accum, ema=None):
    model.train()
    losses = []
    opt.zero_grad()
    pbar = tqdm(loader, desc="train", leave=False)
    for i, batch in enumerate(pbar):
        images = batch["images"].to(device, non_blocking=True)
        intr = batch["intrinsics"].to(device, non_blocking=True)
        c2c = batch["car2cams"].to(device, non_blocking=True)
        gt = batch["gt"].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model(images, intr, c2c)
            loss, parts = criterion(logits, gt)
        loss_for_backward = loss / grad_accum

        if amp_enabled:
            scaler.scale(loss_for_backward).backward()
        else:
            loss_for_backward.backward()

        # Apply optimizer step every `grad_accum` mini-batches
        do_step = ((i + 1) % grad_accum == 0) or ((i + 1) == len(loader))
        if do_step:
            if amp_enabled:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                opt.step()
            sched.step()
            opt.zero_grad()
            if ema is not None:
                ema.update(model)

        losses.append(loss.item())
        pbar.set_postfix({"loss": f"{loss.item():.4f}",
                          "bce": f"{parts['bce']:.3f}",
                          "dice": f"{parts['dice']:.3f}",
                          "lov": f"{parts['lovasz']:.3f}"})
    return float(np.mean(losses))


@torch.no_grad()
def validate(model, loader, device, threshold=0.5, amp_enabled=False):
    model.eval()
    inter_total, union_total = 0, 0
    for batch in tqdm(loader, desc="val", leave=False):
        images = batch["images"].to(device, non_blocking=True)
        intr = batch["intrinsics"].to(device, non_blocking=True)
        c2c = batch["car2cams"].to(device, non_blocking=True)
        gt = batch["gt"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model(images, intr, c2c)
        i, u = iou_binary_batch(logits.float(), gt, threshold=threshold)
        inter_total += i
        union_total += u
    return inter_total / max(union_total, 1)


for epoch in range(cfg["epochs"]):
    t0 = time.time()
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, criterion,
        device, scaler, amp_enabled,
        grad_accum=cfg["grad_accum"], ema=ema,
    )
    history["train_loss"].append(train_loss)

    # Validation
    val_iou = validate(model, val_loader, device, amp_enabled=amp_enabled)
    ema_val_iou = float("nan")
    if ema is not None:
        ema_val_iou = validate(ema.shadow, val_loader, device, amp_enabled=amp_enabled)
    # Official val (для контекста — чтобы видеть разницу с group split)
    official_iou = validate(model, official_val_loader, device, amp_enabled=amp_enabled)

    history["val_iou"].append(val_iou)
    history["ema_val_iou"].append(ema_val_iou)
    history["official_val_iou"].append(official_iou)

    dt = time.time() - t0
    cur_lr = optimizer.param_groups[1]["lr"]
    print(f"Epoch {epoch+1:3d}/{cfg['epochs']}  loss={train_loss:.4f}  "
          f"val_IoU={val_iou:.4f}  ema_val={ema_val_iou:.4f}  "
          f"official_val={official_iou:.4f}  lr_head={cur_lr:.2e}  ({dt:.1f}s)")

    with open(LOG_PATH, "a") as f:
        csv.writer(f).writerow([epoch+1, train_loss, val_iou, ema_val_iou,
                                official_iou, cur_lr, dt])

    # Save last
    torch.save({
        "model": model.state_dict(), "epoch": epoch+1,
        "val_iou": val_iou, "ema_val_iou": ema_val_iou,
        "official_val_iou": official_iou,
        "pos_weight": pos_weight, "history": history, "preset": PRESET,
        "cfg": cfg,
    }, CKPT_LAST)

    # Save best (по group-split val)
    score = max(val_iou, ema_val_iou) if not np.isnan(ema_val_iou) else val_iou
    if score > best_iou:
        best_iou = score
        which = "ema" if (not np.isnan(ema_val_iou) and ema_val_iou >= val_iou) else "model"
        sd = ema.shadow.state_dict() if which == "ema" else model.state_dict()
        torch.save({
            "model": sd, "epoch": epoch+1, "val_iou": score,
            "which": which, "pos_weight": pos_weight,
            "history": history, "preset": PRESET, "cfg": cfg,
        }, CKPT_BEST)
        print(f"  ★ new best: {best_iou:.4f} ({which}) -> {CKPT_BEST}")

    # Save EMA отдельно (для возможного финального инференса)
    if ema is not None:
        torch.save({"model": ema.shadow.state_dict(), "epoch": epoch+1,
                    "val_iou": ema_val_iou, "cfg": cfg}, CKPT_EMA)

print(f"\nFinal best val IoU (group-split): {best_iou:.4f}")

Epoch   1/25  loss=0.6591  val_IoU=0.5425  ema_val=0.4616  official_val=0.4973  lr_head=2.99e-04  (591.9s)
  ★ new best: 0.5425 (model) -> runs/bev_v3_a100_phase2/best.pt


Epoch   2/25  loss=0.6185  val_IoU=0.5603  ema_val=0.4616  official_val=0.5169  lr_head=2.95e-04  (590.2s)
  ★ new best: 0.5603 (model) -> runs/bev_v3_a100_phase2/best.pt


Epoch   3/25  loss=0.6006  val_IoU=0.5703  ema_val=0.4616  official_val=0.5298  lr_head=2.89e-04  (590.5s)
  ★ new best: 0.5703 (model) -> runs/bev_v3_a100_phase2/best.pt


Epoch   4/25  loss=0.5909  val_IoU=0.5740  ema_val=0.4616  official_val=0.5404  lr_head=2.81e-04  (590.0s)
  ★ new best: 0.5740 (model) -> runs/bev_v3_a100_phase2/best.pt


Epoch   5/25  loss=0.5798  val_IoU=0.5744  ema_val=0.4616  official_val=0.5387  lr_head=2.71e-04  (590.4s)
  ★ new best: 0.5744 (model) -> runs/bev_v3_a100_phase2/best.pt


Epoch   6/25  loss=0.5679  val_IoU=0.5624  ema_val=0.4616  official_val=0.4942  lr_head=2.59e-04  (591.0s)


Epoch  23/25  loss=0.4581  val_IoU=0.5829  ema_val=0.5578  official_val=0.5315  lr_head=4.71e-06  (591.1s)


train:  21%|██        | 78/380 [01:31<05:45,  1.14s/it, loss=0.4747, bce=0.430, dice=0.336, lov=0.796]

## 6. Графики

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history["train_loss"], label="train_loss")
axes[0].set_title("train loss"); axes[0].grid(); axes[0].legend()

ax = axes[1]
ax.plot(history["val_iou"], label="val IoU (group-split)", marker="o")
if ema is not None:
    ax.plot(history["ema_val_iou"], label="EMA val IoU", marker="s")
ax.plot(history["official_val_iou"], label="official val IoU", marker="^", linestyle="--", alpha=0.7)
ax.set_title("IoU"); ax.grid(); ax.legend()
plt.tight_layout(); plt.show()

## 7. Threshold sweep на group-split val

Group-split val ближе к test distribution — threshold подобранный на нём должен быть лучше чем на официальном val.

In [7]:
amp_enabled = cfg["use_amp"] and (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
print(f"AMP enabled: {amp_enabled}")

AMP enabled: True


In [ ]:
# Загрузим best.pt для свипа
ckpt = torch.load(CKPT_BEST, map_location=device)
model_for_eval = MultiCamBEVPretrainedEncoder(load_pretrained=False, freeze_encoder=False).to(device)
model_for_eval.load_state_dict(ckpt["model"])
model_for_eval.eval()

all_logits, all_gt = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="sweep group-split val"):
        images = batch["images"].to(device)
        intr = batch["intrinsics"].to(device)
        c2c = batch["car2cams"].to(device)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model_for_eval(images, intr, c2c).float()
        all_logits.append(logits.cpu())
        all_gt.append(batch["gt"])
all_logits = torch.cat(all_logits)
all_gt = torch.cat(all_gt)

best_t, best_score = 0.5, 0.0
for t in np.linspace(0.20, 0.70, 51):
    i, u = iou_binary_batch(all_logits, all_gt, threshold=float(t))
    iou = i / max(u, 1)
    if iou > best_score:
        best_score, best_t = iou, float(t)
print(f"\nBest threshold on group-split val: t={best_t:.3f}  IoU={best_score:.4f}")

# Sanity: пересчитать на official val с тем же threshold
all_logits_o, all_gt_o = [], []
with torch.no_grad():
    for batch in tqdm(official_val_loader, desc="sweep official val"):
        images = batch["images"].to(device)
        intr = batch["intrinsics"].to(device)
        c2c = batch["car2cams"].to(device)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model_for_eval(images, intr, c2c).float()
        all_logits_o.append(logits.cpu())
        all_gt_o.append(batch["gt"])
all_logits_o = torch.cat(all_logits_o)
all_gt_o = torch.cat(all_gt_o)
i, u = iou_binary_batch(all_logits_o, all_gt_o, threshold=best_t)
print(f"Official val IoU at t={best_t}: {i/max(u,1):.4f}")

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
sweep group-split val:  77%|███████▋  | 92/120 [00:48<00:20,  1.35it/s]

[0;31mKernelOutOfMemory[0m: Kernel ran out of memory and has been restarted. If the restart fails, restart the kernel from the Kernel menu.
If the error persists, try choosing a different configuration or optimizing your code.

## 8. Inference на test → submission

In [ ]:
test_ds = BEVDatasetAug(DATA_TEST, mode="test", img_hw=cfg["img_hw"], aug=False)
test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False,
                         num_workers=cfg["num_workers"])
test_info = pd.read_csv(DATA_TEST / "info.csv", index_col=0)
(DATA_TEST / "predicted_static_grids").mkdir(exist_ok=True)

model_for_eval.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="test inference"):
        images = batch["images"].to(device)
        intr = batch["intrinsics"].to(device)
        c2c = batch["car2cams"].to(device)
        idxs = batch["info_idx"]
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model_for_eval(images, intr, c2c).float()
        preds = (torch.sigmoid(logits) > best_t).cpu().numpy().astype(np.int32)
        for k, i in enumerate(idxs):
            out_path = test_info.iloc[i.item()]["predicted_occupancy_grid"]
            grid = preds[k].reshape(1, BEV_H, BEV_W)
            np.save(out_path, grid)
print("all predictions saved")

In [ ]:
# === Robust submission packer (тот же что в train_v1) ===
import zipfile, hashlib

SUBMISSION_NAME = "submission_v3.zip"

missing, shape_bad = [], []
for i, row in test_info.iterrows():
    pred_path = Path(row["predicted_occupancy_grid"])
    if not pred_path.exists():
        missing.append(str(pred_path))
        continue
    arr = np.load(pred_path)
    if arr.shape != (1, BEV_H, BEV_W) or set(np.unique(arr).tolist()) - {0, 1}:
        shape_bad.append(str(pred_path))
if missing or shape_bad:
    raise RuntimeError(f"missing={len(missing)}, shape_bad={len(shape_bad)}")

sub_path = Path(SUBMISSION_NAME)
if sub_path.exists():
    sub_path.unlink()
with zipfile.ZipFile(sub_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(DATA_TEST / "info.csv", arcname="info.csv")
    for npy in sorted((DATA_TEST / "predicted_static_grids").glob("*.npy")):
        zf.write(npy, arcname=f"predicted_static_grids/{npy.name}")
with zipfile.ZipFile(sub_path, "r") as zf:
    assert zf.testzip() is None
    print(f"  zip OK, {len(zf.namelist())} entries")
h = hashlib.sha256()
with open(sub_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print(f"\n=== SUBMISSION ===\n  path: {sub_path.resolve()}\n  size: {sub_path.stat().st_size/1e6:.2f} MB\n  sha256: {h.hexdigest()}")